In [1]:
!pip install -U peft trl

ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [2]:
!pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.5 MB/s eta 0:00:00


In [1]:
import torch
from datasets import load_dataset
from transformers import BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
DATA_PATH = "/content/drive/MyDrive/dataset_final.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/lora-adapter"

In [3]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

In [6]:
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

def format_example(row):
    return {"text": f"[INST] {row['instruction']} [/INST] {row['response']}"}

dataset = dataset.map(format_example)
print(dataset[0])

{'instruction': 'Give three tips for staying healthy.', 'response': 'Maintain a balanced diet with ample fruits and vegetables. Exercise regularly. Keep sleep consistent and sufficient.', 'text': '[INST] Give three tips for staying healthy. [/INST] Maintain a balanced diet with ample fruits and vegetables. Exercise regularly. Keep sleep consistent and sufficient.'}


In [9]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    fp16=False,
    report_to="none",
    dataset_text_field="text",
    max_length=512,
)

In [10]:
trainer = SFTTrainer(
    model=MODEL_NAME,
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
    quantization_config=bnb_config,
)

trainer.train()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Step,Training Loss
5,3.752725
10,1.998632
15,1.869910
20,1.803664
25,1.579130
30,1.394246
35,1.578326
40,1.344004
45,1.312623
50,1.340703


TrainOutput(global_step=75, training_loss=1.5597607676188152, metrics={'train_runtime': 1087.8328, 'train_samples_per_second': 0.552, 'train_steps_per_second': 0.069, 'total_flos': 1872263828275200.0, 'train_loss': 1.5597607676188152, 'epoch': 3.0})

In [11]:
trainer.save_model(OUTPUT_DIR)
print(f"{OUTPUT_DIR}")

/content/drive/MyDrive/lora-adapter


In [13]:
!ls -la {OUTPUT_DIR}
!du -sh {OUTPUT_DIR}

total 30113
drwx------ 5 root root     4096 Sep 16 15:23 .
drwx------ 5 root root     4096 Sep 16 14:22 ..
-rw------- 1 root root     1141 Sep 16 15:23 adapter_config.json
-rw------- 1 root root 27297544 Sep 16 15:23 adapter_model.safetensors
-rw------- 1 root root     1058 Sep 16 15:23 chat_template.jinja
drwx------ 2 root root     4096 Sep 16 15:06 checkpoint-25
drwx------ 2 root root     4096 Sep 16 15:13 checkpoint-50
drwx------ 2 root root     4096 Sep 16 15:19 checkpoint-75
-rw------- 1 root root     1519 Sep 16 15:23 README.md
-rw------- 1 root root      492 Sep 16 15:23 tokenizer_config.json
-rw------- 1 root root  3505608 Sep 16 15:23 tokenizer.json
-rw------- 1 root root     5777 Sep 16 15:23 training_args.bin
275M	/content/drive/MyDrive/lora-adapter
